# Hurst Regimes 02 — roughness of price and volatility (descriptive)

**Discussion:** `discussions/Hurst_Regimes/003_claude_02_roughness_design.md` · **Code:** `src/data_engineering/modeling/hurst_roughness.py`

No trading claim and no gate. All two years of 5-minute history.

1. **Price scaling.** $m_q(\Delta)=\overline{|X_{t+\Delta}-X_t|^q}\propto\Delta^{\zeta(q)}$, with $H=\zeta(2)/2$ fitted separately at 5–30 min, 1–8 h and 1–7 days.
2. **Volatility roughness.** $\log\sigma=\tfrac12\log RV$ on 1h, 4h and 1d blocks (sub-daily blocks deseasonalized).
   - Structure functions and $\zeta(q)$ across $q$; a constant $H_q$ is the monofractal signature.
   - $H$ is reported raw and noise-corrected ($m_2=A\Delta^{2H}+2\eta^2$), then **calibrated by simulation**. Daily RV integration biases $H$ up; RV measurement noise biases it down. Rough-volatility paths with known $H$, the asset's vol-of-vol and the same sample length map estimates back to the true $H$.
3. **Mean reversion of $\log\sigma$.**
   - Autocorrelation: power law vs exponential (OU) decay.
   - fOU fit with the thesis estimators ($\hat\kappa$ from the stationary variance) and its half-life, using the calibrated $H$; the raw COF version is shown for comparison (on log RV, COF is biased upward by block integration).
4. **Stability:** half-year windows.

Environment (optional): `HR_HISTORY_DIR`, `HR_ASSETS`, `HR02_CAL_REPS` (default 3).

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import hurst_regimes as hr
from data_engineering.modeling import hurst_roughness as rg

HIST_DIR = Path(os.getenv('HR_HISTORY_DIR', str(DATA_ROOT / 'history' / 'price_5m')))
ASSETS = [x.strip().upper() for x in os.getenv('HR_ASSETS', 'AVAX,BTC,ETH,HYPE,SOL,TRX,XRP').split(',') if x.strip()]
CAL_REPS = int(os.getenv('HR02_CAL_REPS', str(rg.CAL_REPS)))
OUT = PROJECT_ROOT / 'notebooks' / 'Hurst_Regimes' / '02'
OUT.mkdir(parents=True, exist_ok=True)
print('Hurst Regimes 02', rg.VERSION, rg.selftest())
hist = hr.load_history(HIST_DIR, ASSETS)
LP, EXCLUDED = {}, {}
for a in sorted(hist.underlying.unique()):
    lp = rg.price_grid(hist, a)
    days = int(lp.notna().sum() // 288)
    print(f'{a}: {lp.index[0]} -> {lp.index[-1]} | bars {len(lp):,} | missing {lp.isna().mean():.2%} | {days} days of data')
    if days < rg.MIN_DAYS:
        EXCLUDED[a] = days
    else:
        LP[a] = lp
print('Excluded (fewer than', rg.MIN_DAYS, 'days):', EXCLUDED)

## 1. Price scaling

In [ ]:
price_rows, price_sf = [], []
fig, ax = plt.subplots(figsize=(8, 5))
for a, lp in LP.items():
    sf = rg.structure_function(lp.to_numpy(), rg.PRICE_LAGS)
    price_sf.append(sf.assign(underlying=a))
    for rng_name, (lo, hi) in rg.PRICE_RANGES.items():
        z = rg.zeta(sf, lo, hi)
        price_rows.append({'underlying': a, 'range': rng_name, **{k: z[k] for k in ('H2', 'H1', 'zeta2', 'n_lags')}})
    ax.loglog(sf.lag * 5, sf.m2 / sf.m2.iloc[0] / sf.lag, marker='o', ms=3, label=a)
ax.axhline(1, color='k', lw=0.6)
ax.set_xlabel('lag (minutes)'); ax.set_ylabel('m2(lag) / (lag x m2(5m))   [flat = Brownian]')
ax.set_title('Price: variance ratio vs horizon'); ax.legend(); plt.show()
price_tab = pd.DataFrame(price_rows)
display(price_tab.pivot(index='underlying', columns='range', values='H2')[list(rg.PRICE_RANGES)].round(3))

## 2. Volatility roughness

In [ ]:
vol_est, vol_rows, zeta_rows, vol_sf = {}, [], [], []
for a, lp in LP.items():
    e = rg.vol_H_estimates(lp, deseason=True)
    vol_est[a] = e
    for b, v in e.items():
        vol_rows.append({'underlying': a, 'block': b, 'H_raw': v['raw'], 'H_noise_corr': v['corr'], 'H_cof': v['cof'],
                         'noise_share_lag1': v['nc'].get('noise_share_lag1', np.nan), 'n_blocks': int(v['lv'].notna().sum())})
        zeta_rows.append({'underlying': a, 'block': b, **{f'H_q{q:g}': v['z'].get(f'H{q:g}', np.nan) for q in rg.QS}})
        vol_sf.append(v['sf'].assign(underlying=a, block=b))
vol_tab = pd.DataFrame(vol_rows)
zeta_tab = pd.DataFrame(zeta_rows)
display(vol_tab.round(3))
print('Monofractal check: H_q = zeta(q)/q across q (constant across q = monofractal)')
display(zeta_tab.round(3))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, b in zip(axes, rg.VOL_BLOCKS):
    for a in LP:
        sf = vol_est[a][b]['sf']
        ax.loglog(sf.lag, sf.m2, marker='o', ms=3, label=a)
    ax.set_title(f'log sigma, {b} blocks: m2(lag)'); ax.set_xlabel(f'lag ({b} blocks)')
axes[0].legend(); plt.tight_layout(); plt.show()

## 3. Calibration by simulation (rough-volatility paths with known $H$)

In [ ]:
cal_all, cal_rows = [], []
t0 = time.time()
for a, lp in LP.items():
    d1 = vol_est[a]['1d']['sf']
    nu = float(np.sqrt(d1.loc[d1.lag == 1, 'm2'].iloc[0]))         # daily log-vol increment scale (upper bound: includes noise)
    cal = rg.calibrate(len(lp), nu, reps=CAL_REPS)
    cal_all.append(cal.assign(underlying=a, nu=nu))
    for b in rg.VOL_BLOCKS:
        for kind, col in (('raw', 'H_raw'), ('corr', 'H_noise_corr'), ('cof', 'H_cof')):
            h = float(vol_tab.loc[(vol_tab.underlying == a) & (vol_tab.block == b), col].iloc[0])
            cal_rows.append({'underlying': a, 'block': b, 'kind': kind, 'H_est': h, 'H_calibrated': rg.invert(cal, b, kind, h), 'nu': nu})
    print(f'{a}: calibrated ({time.time() - t0:.0f}s)')
cal_df = pd.concat(cal_all, ignore_index=True)
cal_tab = pd.DataFrame(cal_rows)
display(cal_df[cal_df.underlying == list(LP)[0]].pivot_table(index='H_true', columns=['block', 'kind'], values='H_est').round(3))
display(cal_tab.pivot_table(index='underlying', columns=['block', 'kind'], values='H_calibrated').round(3))

## 4. Mean reversion of $\log\sigma$

In [ ]:
acf_rows, fou_rows, acf_curves = [], [], []
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for a in LP:
    for b in ('1h', '1d'):
        lv = vol_est[a][b]['lv'].to_numpy()
        r = rg.acf(lv, rg.ACF_MAX[b])
        acf_curves.append(pd.DataFrame({'underlying': a, 'block': b, 'lag': np.arange(len(r)), 'acf': r}))
        acf_rows.append({'underlying': a, 'block': b, **rg.decay_fits(r, rg.ACF_MAX[b])})
        ax = axes[0] if b == '1h' else axes[1]
        ax.loglog(np.arange(1, len(r)), np.clip(r[1:], 1e-3, None), label=a)
    for b in ('4h', '1d'):
        hc = float(cal_tab.loc[(cal_tab.underlying == a) & (cal_tab.block == b) & (cal_tab.kind == 'corr'), 'H_calibrated'].iloc[0])
        lv = vol_est[a][b]['lv'].to_numpy()
        fou_rows.append({'underlying': a, 'block': b, 'H_source': 'calibrated', **rg.fou_fit(lv, H=hc)})
        fou_rows.append({'underlying': a, 'block': b, 'H_source': 'cof_raw (thesis)', **rg.fou_fit(lv)})
for ax, b in zip(axes, ('1h', '1d')):
    ax.set_title(f'ACF of log sigma ({b} blocks), log-log: straight = power law'); ax.set_xlabel('lag')
axes[0].legend(); plt.tight_layout(); plt.show()
acf_tab, fou_tab = pd.DataFrame(acf_rows), pd.DataFrame(fou_rows)
display(acf_tab.round(3)); display(fou_tab.round(4))
print('Caveat: kappa = [sigma^2 Gamma(2H+1) / (2 Var)]^(1/(2H)) is extremely sensitive at small H (exponent 1/(2H)); '
      'treat half-lives as order-of-magnitude only. The ACF is the model-free read of mean reversion.')

## 5. Stability (half-year windows)

In [ ]:
stab = []
for a, lp in LP.items():
    half = lp.index.year.astype(str) + np.where(lp.index.month <= 6, '-H1', '-H2')
    for start, g in lp.groupby(half):
        if g.notna().sum() < 288 * 120:
            continue
        sf = rg.structure_function(g.to_numpy(), rg.PRICE_LAGS)
        lv = rg.log_vol_series(g, 288, deseason=False)
        sfv = rg.structure_function(lv.to_numpy(), rg.VOL_LAGS['1d'])
        lh = rg.log_vol_series(g, 48, deseason=True)
        sfh = rg.structure_function(lh.to_numpy(), rg.VOL_LAGS['4h'])
        stab.append({'underlying': a, 'window': str(start), 'H_price_1h_8h': rg.zeta(sf, 12, 96).get('H2'),
                     'H_price_5m_30m': rg.zeta(sf, 1, 6).get('H2'),
                     'H_vol_1d_raw': rg.zeta(sfv, 1, 30).get('H2'), 'H_vol_4h_corr': rg.noise_corrected_H(sfh, 1, 42).get('H_noise_corr')})
stab_tab = pd.DataFrame(stab)
display(stab_tab.round(3))

## 6. Outputs

In [ ]:
price_tab.to_csv(OUT / '02_price_scaling.csv', index=False, float_format='%.5g')
pd.concat(price_sf, ignore_index=True).to_csv(OUT / '02_price_structure.csv', index=False, float_format='%.6g')
vol_tab.merge(zeta_tab, on=['underlying', 'block']).to_csv(OUT / '02_vol_H.csv', index=False, float_format='%.5g')
pd.concat(vol_sf, ignore_index=True).to_csv(OUT / '02_vol_structure.csv', index=False, float_format='%.6g')
cal_df.to_csv(OUT / '02_calibration_map.csv', index=False, float_format='%.5g')
cal_tab.to_csv(OUT / '02_vol_H_calibrated.csv', index=False, float_format='%.5g')
acf_tab.to_csv(OUT / '02_acf_fits.csv', index=False, float_format='%.5g')
pd.concat(acf_curves, ignore_index=True).to_csv(OUT / '02_acf_curves.csv', index=False, float_format='%.5g')
fou_tab.to_csv(OUT / '02_fou_fit.csv', index=False, float_format='%.5g')
stab_tab.to_csv(OUT / '02_stability.csv', index=False, float_format='%.5g')
meta = {'module': '02_roughness', 'version': rg.VERSION, 'assets': list(LP), 'excluded_short_history_days': EXCLUDED, 'cal_reps': CAL_REPS, 'history_dir': str(HIST_DIR),
        'first': str(min(lp.index[0] for lp in LP.values())), 'last': str(max(lp.index[-1] for lp in LP.values()))}
(OUT / '02_run_metadata.json').write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))